# Lecture 3 — Class Exercise
## Line Charts: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
- No spaghetti — if you show multiple lines, grey everything and highlight one
- Remove chart borders, heavy gridlines, rotating text
- Label directly; avoid legends where possible
- Insight title


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('../data/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


## Task 1 - A line chart showing CO2 per Capita for all countries in the dataset

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#EEEEEE), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

In [ ]:
def highlight_line_chart(data, value_col, highlight, color, title, y_title):
    fig = go.Figure()

    # Context lines: all other countries in light grey, thin
    for country, g in data.groupby('Country'):
        if country == highlight:
            continue
        fig.add_trace(go.Scatter(
            x=g['Year'], y=g[value_col], mode='lines',
            line=dict(color='#EEEEEE', width=1),
            hoverinfo='skip', showlegend=False
        ))

    # Highlighted line on top
    h = data[data['Country'] == highlight].sort_values('Year')
    fig.add_trace(go.Scatter(
        x=h['Year'], y=h[value_col], mode='lines',
        line=dict(color=color, width=3.5),
        showlegend=False,
        hovertemplate='%{x}: %{y:,.2f}<extra>' + highlight + '</extra>'
    ))

    # Direct label at end of the line (no legend)
    last = h.iloc[-1]
    fig.add_annotation(
        x=last['Year'], y=last[value_col], text=f"<b>{highlight}</b>",
        showarrow=False, xanchor='left', xshift=8,
        font=dict(color=color, size=14)
    )

    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=20)),
        plot_bgcolor='white', paper_bgcolor='white',
        xaxis=dict(showgrid=False, showline=True, linecolor='#999',
                   tickangle=0, dtick=2, title=None),
        yaxis=dict(showgrid=True, gridcolor='#F2F2F2', gridwidth=0.5,
                   zeroline=False, title=y_title),
        margin=dict(l=60, r=120, t=80, b=50),
        height=500, width=900
    )
    fig.update_xaxes(range=[data['Year'].min(), data['Year'].max() + 1])
    return fig


# Task 1: CO2 per capita, all countries, India highlighted
country = 'India'
h = df[df['Country'] == country].sort_values('Year')
start, end = h['CO2_per_capita'].iloc[0], h['CO2_per_capita'].iloc[-1]
change = (end / start - 1) * 100

fig1 = highlight_line_chart(
    df, 'CO2_per_capita', country, '#D55E00',
    f"{country}'s CO2 per capita changed {change:+.0f}% since 2000 ({start:.2f} \u2192 {end:.2f} t)",
    'CO2 per capita (t)'
)
fig1.show()


---
## Task 2 — A line chart showing CO2 emissions over time for all Asian countries in the dataset, with one country highlighted.

**same requirements as Task1**

In [ ]:
# Task 2: CO2 emissions, Asian countries, China highlighted
asia = df[df['Region'] == 'Asia']

country = 'China'
h = asia[asia['Country'] == country].sort_values('Year')
start, end = h['CO2_Mt'].iloc[0], h['CO2_Mt'].iloc[-1]

fig2 = highlight_line_chart(
    asia, 'CO2_Mt', country, '#0072B2',
    f"{country}'s emissions grew {end/start:.1f}x since 2000, dwarfing every other Asian country in the dataset",
    'CO2 emissions (Mt)'
)
fig2.show()
